In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix

# =====================
# Config
# =====================
DATASET_DIR = "dataset"
BATCH_SIZE = 32
NUM_EPOCHS = 10
LR = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =====================
# Transforms
# =====================
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# =====================
# Datasets & Loaders
# =====================
train_dataset = datasets.ImageFolder(
    root=r"C:\Users\Acer\Desktop\gp\datasets\images\train",
    transform=train_transform
)



val_dataset = datasets.ImageFolder(
    root=r"C:\Users\Acer\Desktop\gp\datasets\images\val",
    transform=val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Class mapping:", train_dataset.class_to_idx)

# =====================
# Model
# =====================
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# replace last layer
model.fc = nn.Linear(model.fc.in_features, 2)

model = model.to(DEVICE)

# =====================
# Loss & Optimizer
# =====================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# =====================
# Training Loop
# =====================
for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")

    # ---- Train ----
    model.train()
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, preds = torch.max(outputs, 1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_acc = train_correct / train_total
    print(f"Train Accuracy: {train_acc:.4f}")

    y_true = []
    y_pred = []
    

    fp_indices = []      # index تصاویر FP


    model.eval()
    val_correct = 0
    val_total = 0

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(val_loader):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        val_correct += (preds == labels).sum().item()
        val_total += labels.size(0)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

        # محاسبه index واقعی در dataset
        batch_start_idx = batch_idx * val_loader.batch_size

        for i in range(len(labels)):
            true_label = labels[i].item()
            pred_label = preds[i].item()

            # FP: noglasses (1) -> glasses (0)
            if true_label == 1 and pred_label == 0:
                fp_indices.append(batch_start_idx + i)





Class mapping: {'glasses': 0, 'noglasses': 1}

Epoch 1/10
Train Accuracy: 0.7653

Epoch 2/10
Train Accuracy: 0.9906

Epoch 3/10
Train Accuracy: 1.0000

Epoch 4/10
Train Accuracy: 1.0000

Epoch 5/10
Train Accuracy: 1.0000

Epoch 6/10
Train Accuracy: 1.0000

Epoch 7/10
Train Accuracy: 1.0000

Epoch 8/10
Train Accuracy: 1.0000

Epoch 9/10
Train Accuracy: 1.0000

Epoch 10/10
Train Accuracy: 1.0000


In [2]:
# Accuracy
val_acc = val_correct / val_total
print(f"Val Accuracy: {val_acc:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)

print(f"Number of FP: {len(fp_indices)}")

print("False Positive image paths:")
for idx in fp_indices:
    path, _ = val_dataset.samples[idx]
    print(path)


# =====================
# Save Model
# =====================
torch.save(model.state_dict(), "resnet18_glasses.pth")
print("\nModel saved as resnet18_glasses.pth")


Val Accuracy: 0.9512
Confusion Matrix:
[[42  2]
 [ 2 36]]
Number of FP: 2
False Positive image paths:
C:\Users\Acer\Desktop\gp\datasets\images\val\noglasses\pexels-photo-1858175.jpeg
C:\Users\Acer\Desktop\gp\datasets\images\val\noglasses\pexels-photo-8127811.jpeg

Model saved as resnet18_glasses.pth
